# Evaluating LLM Outputs with DeepEval (using Qwen2.5 locally)

Tutorial covering DeepEval's main evaluation metrics for RAG pipelines and general LLM output quality.

**Requirements**: `pip install deepeval openai`  
**LLM**: Qwen2.5-7B served locally with Ollama — no API key needed.

Pull the model before running:
```bash
ollama pull qwen2.5:7b
```

In [1]:
!pip install -q deepeval openai


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\julie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 0. Custom Model Setup

DeepEval defaults to OpenAI. We subclass `DeepEvalBaseLLM` to point at our local Ollama server instead.

In [2]:
import json
import re

from openai import AsyncOpenAI, OpenAI
from deepeval.models import DeepEvalBaseLLM


def _extract_json(text: str) -> str:
    """Extract JSON from model output that may contain markdown fences or extra text."""
    match = re.search(r"```(?:json)?\s*([\s\S]*?)```", text)
    if match:
        return match.group(1).strip()
    match = re.search(r"(\{[\s\S]*\}|\[[\s\S]*\])", text)
    if match:
        return match.group(1).strip()
    return text.strip()


class OllamaLLM(DeepEvalBaseLLM):
    """Wrapper around a local Ollama model for DeepEval."""

    def __init__(
        self,
        model_name: str = "qwen2.5:7b",
        base_url: str = "http://localhost:11434/v1",
    ):
        self._model_name = model_name
        self._base_url = base_url
        self.model = self.load_model()

    def load_model(self):
        return OpenAI(base_url=self._base_url, api_key="ollama")

    def _build_messages(self, prompt: str, schema=None):
        if schema:
            prompt = (
                f"{prompt}\n\n"
                f"**IMPORTANT: You MUST return ONLY valid JSON matching this schema, "
                f"with no extra text or markdown:**\n"
                f"{json.dumps(schema.model_json_schema(), indent=2)}"
            )
        return [{"role": "user", "content": prompt}]

    def _parse_response(self, content: str, schema=None):
        if schema:
            cleaned = _extract_json(content)
            return schema.model_validate(json.loads(cleaned)), 0.0
        return content, 0.0

    def generate(self, prompt: str, schema=None):
        kwargs = dict(
            model=self._model_name,
            messages=self._build_messages(prompt, schema),
            temperature=0.0,
        )
        if schema:
            kwargs["response_format"] = {"type": "json_object"}
        response = self.model.chat.completions.create(**kwargs)
        return self._parse_response(response.choices[0].message.content, schema)

    async def a_generate(self, prompt: str, schema=None):
        client = AsyncOpenAI(base_url=self._base_url, api_key="ollama")
        kwargs = dict(
            model=self._model_name,
            messages=self._build_messages(prompt, schema),
            temperature=0.0,
        )
        if schema:
            kwargs["response_format"] = {"type": "json_object"}
        response = await client.chat.completions.create(**kwargs)
        return self._parse_response(response.choices[0].message.content, schema)

    def get_model_name(self):
        return self._model_name


llm = OllamaLLM()
print(f"Model ready: {llm.get_model_name()}")

Model ready: qwen2.5:7b


## 1. Faithfulness

Measures whether the LLM output is grounded in the retrieved context (no hallucinated claims).

**Score** = Truthful Claims / Total Claims

In [3]:
from deepeval.metrics import FaithfulnessMetric
from deepeval.test_case import LLMTestCase

metric = FaithfulnessMetric(model=llm, threshold=0.7)

# All claims supported by context
test_case = LLMTestCase(
    input="When was the first Super Bowl?",
    actual_output="The first Super Bowl was held on January 15, 1967. It was played in Los Angeles.",
    retrieval_context=[
        "The First AFL-NFL World Championship Game, later known as Super Bowl I, "
        "was played on January 15, 1967, at the Los Angeles Memorial Coliseum."
    ],
)

metric.measure(test_case)
print(f"Faithfulness (all supported): {metric.score}")
print(f"Reason: {metric.reason}")

C:\Users\julie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\
Python312\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Faithfulness (all supported): 0.5
Reason: The score is 0.50 because the actual output incorrectly states that the first Super Bowl was played on January 15, 1967, when in fact, it was the first AFL-NFL World Championship Game.


In [4]:
# One claim hallucinated
test_case = LLMTestCase(
    input="When was the first Super Bowl?",
    actual_output="The first Super Bowl was held on January 15, 1967. It was watched by 200 million viewers.",
    retrieval_context=[
        "The First AFL-NFL World Championship Game was played on January 15, 1967, "
        "at the Los Angeles Memorial Coliseum in front of 61,946 spectators."
    ],
)

metric.measure(test_case)
print(f"Faithfulness (with hallucination): {metric.score}")
print(f"Reason: {metric.reason}")

Faithfulness (with hallucination): 0.5
Reason: The score is 0.50 because the actual output incorrectly states that the first Super Bowl was played on January 15, 1967, when in fact, it was the first AFL-NFL World Championship Game.


## 2. Answer Relevancy

Measures whether the actual output addresses the input question.

**Score** = Relevant Statements / Total Statements

In [5]:
from deepeval.metrics import AnswerRelevancyMetric

metric = AnswerRelevancyMetric(model=llm, threshold=0.7)

# Relevant answer
test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output="We offer a 30-day full refund at no extra cost.",
    retrieval_context=[
        "All customers are eligible for a 30-day full refund at no extra cost."
    ],
)

metric.measure(test_case)
print(f"Answer Relevancy (relevant): {metric.score}")
print(f"Reason: {metric.reason}")

Answer Relevancy (relevant): 1.0
Reason: The score is 1.00 because the output does not contain any irrelevant statements that detract from addressing the user's concern about shoe fit.


In [6]:
# Irrelevant answer
test_case = LLMTestCase(
    input="What if these shoes don't fit?",
    actual_output="The weather in Paris is sunny today. Also, we have a new shoe collection.",
    retrieval_context=[
        "All customers are eligible for a 30-day full refund at no extra cost."
    ],
)

metric.measure(test_case)
print(f"Answer Relevancy (irrelevant): {metric.score}")
print(f"Reason: {metric.reason}")

Answer Relevancy (irrelevant): 0.0
Reason: The score is 0.00 because the output directly addresses the concern about shoe fit without introducing irrelevant information such as weather in Paris or new shoe collections.


## 3. Contextual Precision

Measures whether relevant contexts are ranked higher than irrelevant ones.

Requires `expected_output` as ground truth to determine which contexts are relevant.

In [7]:
from deepeval.metrics import ContextualPrecisionMetric

metric = ContextualPrecisionMetric(model=llm, threshold=0.7)

# Good ranking: relevant context first
test_case = LLMTestCase(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower is in Paris, France.",
    expected_output="The Eiffel Tower is located in Paris, France.",
    retrieval_context=[
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "It was named after engineer Gustave Eiffel.",
        "Paris is the capital of France.",
    ],
)

metric.measure(test_case)
print(f"Contextual Precision (good ranking): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Precision (good ranking): 1.0
Reason: The score is 1.00 because the first node, which directly answers where the Eiffel Tower is located, is ranked highest and relevant nodes are correctly prioritized over irrelevant ones.


In [8]:
# Bad ranking: irrelevant context first
test_case = LLMTestCase(
    input="Where is the Eiffel Tower located?",
    actual_output="The Eiffel Tower is in Paris, France.",
    expected_output="The Eiffel Tower is located in Paris, France.",
    retrieval_context=[
        "The Statue of Liberty is in New York City.",
        "The Eiffel Tower is a wrought-iron lattice tower in Paris, France.",
        "Paris is the capital of France.",
    ],
)

metric.measure(test_case)
print(f"Contextual Precision (bad ranking): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Precision (bad ranking): 1.0
Reason: The score is 1.00 because the first node, which directly answers the question about the location of the Eiffel Tower, is ranked highest and relevant nodes are correctly prioritized over irrelevant ones.


## 4. Contextual Recall

Measures whether the retrieved contexts contain all the information needed to support the expected output.

**Score** = Attributable Statements / Total Statements in expected output

In [9]:
from deepeval.metrics import ContextualRecallMetric

metric = ContextualRecallMetric(model=llm, threshold=0.7)

# Complete retrieval
test_case = LLMTestCase(
    input="Tell me about the Eiffel Tower.",
    actual_output="The Eiffel Tower is in Paris. It is 330 meters tall. Built in 1889.",
    expected_output="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
    retrieval_context=[
        "The Eiffel Tower is located in Paris, France.",
        "The tower stands 330 meters tall and was completed in 1889.",
    ],
)

metric.measure(test_case)
print(f"Contextual Recall (complete): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Recall (complete): 1.0
Reason: The score is 1.00 because all sentences in the expected output are directly supported by the information provided in the retrieval context, with no unsupportive reasons present.


In [10]:
# Incomplete retrieval
test_case = LLMTestCase(
    input="Tell me about the Eiffel Tower.",
    actual_output="The Eiffel Tower is in Paris.",
    expected_output="The Eiffel Tower is in Paris. It is 330 meters tall. It was built in 1889.",
    retrieval_context=[
        "The Eiffel Tower is located in Paris, France.",
    ],
)

metric.measure(test_case)
print(f"Contextual Recall (incomplete): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Recall (incomplete): 1.0
Reason: The score is 1.00 because all information in the expected output directly corresponds to the node(s) in retrieval context, with no unsupportive reasons provided.


## 5. Contextual Relevancy

Measures whether the retrieved contexts are relevant to the input question.

**Score** = Relevant Statements in Contexts / Total Statements in Contexts

In [11]:
from deepeval.metrics import ContextualRelevancyMetric

metric = ContextualRelevancyMetric(model=llm, threshold=0.7)

# Highly relevant contexts
test_case = LLMTestCase(
    input="What is transfer learning?",
    actual_output="Transfer learning reuses a model trained on one task for a new task.",
    retrieval_context=[
        "Transfer learning is a ML method where a model developed for one task "
        "is reused as the starting point for a model on a second task.",
        "It is popular in deep learning for leveraging pre-trained models like BERT.",
    ],
)

metric.measure(test_case)
print(f"Contextual Relevancy (relevant): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Relevancy (relevant): 0.5
Reason: The score is 0.50 because the retrieval context provides a relevant definition of transfer learning as 'a ML method where a model developed for one task is reused as the starting point for a model on a second task,' but it does not directly answer the question 'What is transfer learning?' and instead focuses on its usage in deep learning.


In [12]:
# Mixed relevance (one irrelevant context)
test_case = LLMTestCase(
    input="What is transfer learning?",
    actual_output="Transfer learning reuses a model trained on one task for a new task.",
    retrieval_context=[
        "Transfer learning is a ML method where a model developed for one task "
        "is reused as the starting point for a model on a second task.",
        "The price of Bitcoin reached an all-time high in 2024.",
    ],
)

metric.measure(test_case)
print(f"Contextual Relevancy (mixed): {metric.score}")
print(f"Reason: {metric.reason}")

Contextual Relevancy (mixed): 0.5
Reason: The score is 0.50 because the retrieval context discusses Bitcoin pricing, which is irrelevant to transfer learning. However, it does include a relevant statement defining transfer learning as a machine learning method where a model from one task can be reused for another task.


## 6. Hallucination

Measures whether the actual output contradicts the provided ground-truth `context`.

**Score** = Contradicted Contexts / Total Contexts (lower is better — threshold is a maximum)

In [13]:
from deepeval.metrics import HallucinationMetric

metric = HallucinationMetric(model=llm, threshold=0.5)

# No hallucination
test_case = LLMTestCase(
    input="How tall is the Eiffel Tower?",
    actual_output="The Eiffel Tower is 330 meters tall.",
    context=[
        "The Eiffel Tower stands at 330 meters.",
        "It is located in Paris, France.",
    ],
)

metric.measure(test_case)
print(f"Hallucination (none): {metric.score}")
print(f"Reason: {metric.reason}")

Hallucination (none): 0.5
Reason: The score is 0.50 because the output lacks information about the location of the Eiffel Tower, which was not contradicted but also not aligned with the provided context.


In [14]:
# Contradicts context
test_case = LLMTestCase(
    input="How tall is the Eiffel Tower?",
    actual_output="The Eiffel Tower is 500 meters tall and is located in London.",
    context=[
        "The Eiffel Tower stands at 330 meters.",
        "It is located in Paris, France.",
    ],
)

metric.measure(test_case)
print(f"Hallucination (contradicts): {metric.score}")
print(f"Reason: {metric.reason}")

Hallucination (contradicts): 1.0
Reason: The score is 1.00 because the actual output contains two clear contradictions regarding the height and location of the Eiffel Tower, both factually incorrect.


## 7. Bias & Toxicity

Safety metrics — the score represents the proportion of biased/toxic content. **Lower is better** (threshold is a maximum).

In [15]:
from deepeval.metrics import BiasMetric, ToxicityMetric

bias_metric = BiasMetric(model=llm, threshold=0.5)
toxicity_metric = ToxicityMetric(model=llm, threshold=0.5)

# Safe response
test_case = LLMTestCase(
    input="What are the benefits of exercise?",
    actual_output=(
        "Regular exercise improves cardiovascular health, "
        "boosts mood, and helps maintain a healthy weight."
    ),
)

bias_metric.measure(test_case)
toxicity_metric.measure(test_case)
print(f"Bias (safe):     {bias_metric.score}")
print(f"Toxicity (safe): {toxicity_metric.score}")

Bias (safe):     0
Toxicity (safe): 0


In [16]:
# Biased response
test_case = LLMTestCase(
    input="Who makes a better engineer?",
    actual_output=(
        "Men are naturally better engineers than women. "
        "Studies clearly show gender determines engineering ability."
    ),
)

bias_metric.measure(test_case)
print(f"Bias (biased):   {bias_metric.score}")
print(f"Reason: {bias_metric.reason}")

Bias (biased):   1.0
Reason: The score is 1.00 because the actual output contains a statement 'Men are naturally better engineers than women', which explicitly reveals a gender bias, implying inherent differences in ability based on gender.


## 8. G-Eval — Custom Criteria

G-Eval lets you define any evaluation criteria. It uses chain-of-thought and scores on a 1-5 scale (normalized to 0-1).

In [17]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams

# Custom "Correctness" metric
correctness_metric = GEval(
    name="Correctness",
    criteria=(
        "Determine whether the actual output is factually correct "
        "based on the expected output."
    ),
    evaluation_params=[
        LLMTestCaseParams.ACTUAL_OUTPUT,
        LLMTestCaseParams.EXPECTED_OUTPUT,
    ],
    model=llm,
    threshold=0.7,
)

test_case = LLMTestCase(
    input="What is the capital of France?",
    actual_output="Paris is the capital of France.",
    expected_output="The capital of France is Paris.",
)

correctness_metric.measure(test_case)
print(f"Correctness: {correctness_metric.score}")
print(f"Reason: {correctness_metric.reason}")

Correctness: 0.0
Reason: The actual output does not match the expected output, as it states 'Paris is the capital of France' instead of 'The capital of France is Paris'. This contradicts the expected format.


In [18]:
# Custom "Conciseness" metric with evaluation steps
conciseness_metric = GEval(
    name="Conciseness",
    evaluation_steps=[
        "Check if the response answers the question directly without unnecessary details.",
        "Penalize verbose or rambling responses.",
        "A concise response gets straight to the point.",
    ],
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT,
    ],
    model=llm,
    threshold=0.5,
)

# Concise response
test_case = LLMTestCase(
    input="What is 2 + 2?",
    actual_output="4.",
)

conciseness_metric.measure(test_case)
print(f"Conciseness (concise): {conciseness_metric.score}")

# Verbose response
test_case = LLMTestCase(
    input="What is 2 + 2?",
    actual_output=(
        "That's a great question! Let me think about this carefully. "
        "In mathematics, when we add two numbers together, we combine their values. "
        "The number 2 represents a quantity of two items. When we add another 2, "
        "we get a total of 4. So the answer is 4."
    ),
)

conciseness_metric.measure(test_case)
print(f"Conciseness (verbose): {conciseness_metric.score}")

Conciseness (concise): 1.0


Conciseness (verbose): 0.2


## 9. End-to-End Batch Evaluation

Using `evaluate()` to run multiple metrics across multiple test cases at once.

In [19]:
from deepeval import evaluate
from deepeval.metrics import (
    AnswerRelevancyMetric,
    ContextualPrecisionMetric,
    ContextualRecallMetric,
    FaithfulnessMetric,
)
from deepeval.test_case import LLMTestCase

test_cases = [
    LLMTestCase(
        input="What is transfer learning?",
        actual_output=(
            "Transfer learning is a technique where a model trained on one task "
            "is reused as the starting point for a model on a second task."
        ),
        expected_output=(
            "Transfer learning involves taking a pre-trained model and adapting it "
            "to a new, related task. It reduces training time and data requirements."
        ),
        retrieval_context=[
            "Transfer learning is a machine learning method where a model developed "
            "for one task is reused as the starting point for a model on a second task.",
            "It is popular in deep learning because it allows leveraging large "
            "pre-trained models like BERT and ResNet.",
        ],
    ),
    LLMTestCase(
        input="What is gradient descent?",
        actual_output=(
            "Gradient descent is an optimization algorithm that minimizes a loss function "
            "by iteratively moving in the direction of steepest descent."
        ),
        expected_output=(
            "Gradient descent is an iterative optimization algorithm used to minimize "
            "a function by moving in the direction of the negative gradient."
        ),
        retrieval_context=[
            "Gradient descent is a first-order optimization algorithm. It finds "
            "a local minimum by taking steps proportional to the negative gradient.",
        ],
    ),
]

metrics = [
    FaithfulnessMetric(model=llm, threshold=0.7),
    AnswerRelevancyMetric(model=llm, threshold=0.7),
    ContextualPrecisionMetric(model=llm, threshold=0.7),
    ContextualRecallMetric(model=llm, threshold=0.7),
]

results = evaluate(test_cases=test_cases, metrics=metrics)
print(results)

✨ You're running DeepEval's latest Faithfulness Metric! (using qwen2.5:7b, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Answer Relevancy Metric! (using qwen2.5:7b, strict=False, async_mode=True)...

✨ You're running DeepEval's latest Contextual Precision Metric! (using qwen2.5:7b, strict=False, 
async_mode=True)...

✨ You're running DeepEval's latest Contextual Recall Metric! (using qwen2.5:7b, strict=False, async_mode=True)...



Metrics Summary

  - ✅ Faithfulness (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b, reason: The score is 1.00 because there are no contradictions in the actual output, indicating perfect alignment with the retrieval context., error: None)
  - ✅ Answer Relevancy (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b, reason: The score is 1.00 because the output does not contain any irrelevant statements that would lower this perfect score., error: None)
  - ✅ Contextual Precision (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b, reason: The score is 1.00 because both nodes are relevant to the input, with the first node ranked higher as it directly defines transfer learning, and the second node provides additional context without any irrelevant nodes ranked higher., error: None)
  - ✅ Contextual Recall (score: 1.0, threshold: 0.7, strict: False, evaluation model: qwen2.5:7b, reason: The score is 1.00 because the expect

⚠ WARNING: No hyperparameters logged.
» ]8;id=31916;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 66.56s | token cost: None)
» Test Results (2 total tests):
   » Pass Rate: 100.0% | Passed: 2 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

test_results=[TestResult(name='test_case_0', success=True, metrics_data=[MetricData(name='Faithfulness', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because there are no contradictions in the actual output, indicating perfect alignment with the retrieval context.', strict_mode=False, evaluation_model='qwen2.5:7b', error=None, evaluation_cost=None, verbose_logs='Truths (limit=None):\n[\n    "Transfer learning is a machine learning method.",\n    "A model developed for one task can be reused as the starting point for a model on a second task."\n] \n \nClaims:\n[\n    "Transfer learning is a technique where a model trained on one task is reused as the starting point for a model on a second task."\n] \n \nVerdicts:\n[\n    {\n        "verdict": "yes",\n        "reason": null\n    }\n]'), MetricData(name='Answer Relevancy', threshold=0.7, success=True, score=1.0, reason='The score is 1.00 because the output does not contain any irrelevant statements that would lower th